# Project 02: HVAC export cable for an offshore wind farm

**Tools:** pandapower, Python, matplotlib
**Network data:** pandapower standard line type `N2XS(FL)2Y 1x300 RM/35 64/110 kV`
**Notebook status:** executed, all numbers below are produced by this notebook

---

## The question

> A 200 MW offshore wind farm needs an AC export connection to shore. How much shunt
> reactive compensation does the cable need, and at what distance does the AC option
> stop working at all?

A long AC submarine cable behaves as a large distributed capacitance. Even with no
power flowing it draws charging current, and that current consumes conductor capacity
and pushes up the voltage at the offshore end. This notebook quantifies both effects
and finds the length at which no amount of shunt compensation keeps the offshore
voltage inside the operating band.

## Why this problem

I wanted a problem where the physics has a clean answer and the answer is a number a
developer would actually use. Reactive compensation is also the reason HVDC exists for
long offshore connections, so working through the AC case is the way to understand why
the industry switches technology at a certain distance.


## 1. Setup and cable data

The cable parameters are not invented. They come from pandapower's standard type
library. The provenance is documented in `pandapower/std_types.py`, in the function
`basic_line_std_types`, whose comment attributes the cable block to Heuck, Dettmann
and Schulz (2013), page 744.


In [ ]:
import json
import numpy as np, pandas as pd
import pandapower as pp
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = "/tmp/work/out/"
CABLE = "N2XS(FL)2Y 1x300 RM/35 64/110 kV"
V_KV, F_HZ, FARM_MW = 110.0, 50.0, 200.0
V_MIN, V_MAX, N_CIRC = 0.94, 1.06, 2

_n = pp.create_empty_network()
T = pp.load_std_type(_n, CABLE, element="line")
S_CIRC = float(np.sqrt(3) * V_KV * T["max_i_ka"])

print(f"R {T['r_ohm_per_km']} ohm/km, X {T['x_ohm_per_km']} ohm/km, "
      f"C {T['c_nf_per_km']} nF/km, Imax {T['max_i_ka']} kA")
print(f"single circuit rating {S_CIRC:.1f} MVA")
print(f"circuits needed for {FARM_MW:.0f} MW: {int(np.ceil(FARM_MW/S_CIRC))}")

A single circuit is rated 112.0 MVA, so a 200 MW farm
needs **2 parallel circuits**. That is the first design decision and it
falls straight out of the conductor rating.

## 2. Charging reactive power

For a cable of length `l` the shunt capacitance is `C = c' * l`, and at nominal voltage
the reactive power it generates is

$$Q_c = V^2 \, \omega \, C \, l$$

This is the standard result for a lossless shunt capacitance and it scales linearly with
length.


In [ ]:
def q_charging(L, n=N_CIRC):
    """No-load charging reactive power, Mvar.  Q = V^2 * w * C * l"""
    c_f = T["c_nf_per_km"] * 1e-9 * L * n
    return (V_KV * 1e3) ** 2 * 2 * np.pi * F_HZ * c_f / 1e6

for L in (50, 100, 150, 200):
    q = q_charging(L)
    print(f"{L:3d} km: {q:6.1f} Mvar  ({100*q/FARM_MW:.0f}% of farm rating)")

At 100 km the two circuits generate **109.5 Mvar**, which is
55 percent of the farm's active power rating. That is
the number that makes long AC connections difficult.

## 3. The network model

Two buses, the offshore collector and the onshore connection point, joined by the export
circuits. The wind farm is a static generator at unity power factor. Shunt reactors are
created at both ends with zero rating and set later.

One detail that cost me time: in pandapower a shunt with **positive** `q_mvar` absorbs
reactive power. I first wrote it negative, which made the reactors behave as capacitors
and pushed the voltage the wrong way. The symptom was compensation making the voltage
problem worse, which is a useful thing to have debugged once.


In [ ]:
def make(L, p_mw=FARM_MW, n=N_CIRC):
    """Offshore bus, export cables, onshore slack. Shunts created at zero."""
    net = pp.create_empty_network(f_hz=F_HZ)
    b0 = pp.create_bus(net, vn_kv=V_KV, name="offshore")
    b1 = pp.create_bus(net, vn_kv=V_KV, name="onshore")
    pp.create_ext_grid(net, b1, vm_pu=1.0)
    for k in range(n):
        pp.create_line(net, b0, b1, length_km=L, std_type=CABLE, name=f"circuit_{k}")
    pp.create_sgen(net, b0, p_mw=p_mw, q_mvar=0.0, name="wind farm")
    pp.create_shunt(net, b0, q_mvar=0.0, p_mw=0.0, name="offshore reactor")
    pp.create_shunt(net, b1, q_mvar=0.0, p_mw=0.0, name="onshore reactor")
    return net

def solve(net, L, ratio):
    """Set compensation to `ratio` of charging Mvar, split evenly, then solve."""
    q = q_charging(L) * ratio / 2.0
    net.shunt.loc[:, "q_mvar"] = [q, q]   # positive q_mvar = absorbing = shunt reactor
    try:
        pp.runpp(net, numba=False)
        return dict(converged=True,
            v_off=float(net.res_bus.vm_pu.iloc[0]),
            loading=float(net.res_line.loading_percent.max()),
            p_grid=float(-net.res_ext_grid.p_mw.iloc[0]),
            q_grid=float(-net.res_ext_grid.q_mvar.iloc[0]),
            losses=float(net.res_line.pl_mw.sum()))
    except Exception:
        return dict(converged=False, v_off=np.nan, loading=np.nan,
                    p_grid=np.nan, q_grid=np.nan, losses=np.nan)

net = make(100)
print(solve(net, 100, 0.0))

## 4. Minimum compensation by bisection

For each length I find the smallest compensation ratio that keeps the offshore voltage
within 0.94 to 1.06 per unit and the circuits under 100 percent loading. A linear scan
would need dozens of power flows per length, so I bisect instead: eight iterations gives
a resolution of about 0.4 percent, and the whole sweep runs in seconds.


In [ ]:
def inband(s):
    return s["converged"] and V_MIN <= s["v_off"] <= V_MAX and s["loading"] <= 100.0

    lengths = np.arange(10, 210, 10)
    rows_len, rows_min = [], []
    for L in lengths:
        net = make(L)
        s0 = solve(net, L, 0.0)                              # uncompensated
        rows_len.append(dict(length_km=int(L), q_charge_mvar=q_charging(L),
                             q_pct_of_farm=100*q_charging(L)/FARM_MW,
                             **{f"unc_{k}": v for k, v in s0.items()}))
        # minimum compensation, by bisection
        if inband(s0):
            r_min = 0.0
        else:
            sf = solve(net, L, 1.0)
            if not inband(sf):
                rows_min.append(dict(length_km=int(L), min_comp_ratio=np.nan,
                                     q_comp_mvar=np.nan, feasible=False, **sf)); continue
            lo, hi = 0.0, 1.0
            for _ in range(8):
                mid = (lo + hi) / 2
                if inband(solve(net, L, mid)): hi = mid
                else: lo = mid
            r_min = hi
        s = solve(net, L, r_min)
        rows_min.append(dict(length_km=int(L), min_comp_ratio=r_min,
                             q_comp_mvar=q_charging(L)*r_min, feasible=True, **s))
    df_len, df_min = pd.DataFrame(rows_len), pd.DataFrame(rows_min)

## 5. Results

| Quantity | Value |
|---|---|
| Cable | `N2XS(FL)2Y 1x300 RM/35 64/110 kV` |
| Circuit rating | 112.0 MVA |
| Circuits for 200 MW | 2 |
| Charging power at 100 km | 109.5 Mvar |
| No compensation needed up to | 80 km |
| Compensation needed at 100 km | 46 percent, 50.5 Mvar |
| Longest feasible AC length | **160 km** |

**Key finding.** The connection needs no compensation at all out to
80 km. From there the required compensation
climbs steeply, reaching essentially full compensation at 160 km.
Past 160 km the offshore voltage leaves the band even with the
charging power fully cancelled, because the remaining problem is the series impedance
rather than the shunt capacitance. That is the point where the AC option stops being a
design choice and HVDC becomes the only answer.

**A second finding I did not expect.** Compensation reduces losses. At 100 km the losses
fall from 9.42 MW uncompensated to 9.07 MW
compensated, and circuit loading falls from 95.2 percent to
90.2 percent. The reactors are not only a voltage control measure,
they free up thermal capacity by removing reactive current from the conductor.


In [ ]:
    L_REF, net = 100, make(100)
    df_comp = pd.DataFrame([dict(comp_ratio=r, q_comp_mvar=q_charging(L_REF)*r,
                                 **solve(net, L_REF, r)) for r in np.arange(0, 1.001, 0.05)])

    for d, n in ((df_len,"length_sweep"),(df_comp,"compensation_sweep"),(df_min,"min_compensation")):
        d.to_csv(OUT+f"p02_{n}.csv", index=False)

    g = lambda df, L, c: float(df.loc[df.length_km == L, c].iloc[0])
    feas = df_min[df_min.feasible]
    summary = dict(cable=CABLE, r_ohm_per_km=float(T["r_ohm_per_km"]),
        x_ohm_per_km=float(T["x_ohm_per_km"]), c_nf_per_km=float(T["c_nf_per_km"]),
        max_i_ka=float(T["max_i_ka"]), circuit_rating_mva=S_CIRC, n_circuits=N_CIRC,
        farm_mw=FARM_MW,
        q_charge_50km=q_charging(50), q_charge_100km=q_charging(100), q_charge_200km=q_charging(200),
        q_charge_100km_pct_of_farm=100*q_charging(100)/FARM_MW,
        first_length_needing_compensation=int(df_min[df_min.min_comp_ratio.fillna(1) > 0].length_km.min()),
        max_feasible_length_km=int(feas.length_km.max()) if len(feas) else None,
        min_comp_100km=g(df_min,100,"min_comp_ratio"),
        q_comp_100km_mvar=g(df_min,100,"q_comp_mvar"),
        v_off_100km_uncomp=g(df_len,100,"unc_v_off"), v_off_100km_comp=g(df_min,100,"v_off"),
        losses_100km_uncomp_mw=g(df_len,100,"unc_losses"), losses_100km_comp_mw=g(df_min,100,"losses"),
        loading_100km_uncomp=g(df_len,100,"unc_loading"), loading_100km_comp=g(df_min,100,"loading"))
    json.dump(summary, open(OUT+"p02_summary.json","w"), indent=1)
    print(json.dumps(summary, indent=1), flush=True)
    print("\n", df_min[["length_km","min_comp_ratio","q_comp_mvar","v_off","loading","losses"]].to_string(index=False))

## 6. Figures

In [ ]:
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
    ax[0].plot(df_len.length_km, df_len.q_charge_mvar, color="#4B7FA8", lw=2)
    ax[0].axhline(FARM_MW, ls="--", c="#C4714A", lw=1.2)
    ax[0].annotate(f"{FARM_MW:.0f} MW farm rating", (15, FARM_MW*1.04), fontsize=9, color="#C4714A")
    ax[0].set(xlabel="Cable length [km]", ylabel="Charging reactive power [Mvar]",
              title="Charging power scales with length")
    ax[1].plot(df_comp.comp_ratio*100, df_comp.v_off, color="#4A7C6F", lw=2)
    ax[1].axhline(V_MAX, ls="--", c="#C4714A", lw=1); ax[1].axhline(V_MIN, ls="--", c="#C4714A", lw=1)
    ax[1].set(xlabel="Compensation [% of charging Mvar]", ylabel="Offshore voltage [p.u.]",
              title=f"Voltage vs compensation, {L_REF} km")
    ax[2].plot(feas.length_km, feas.min_comp_ratio*100, "o-", color="#B89E7E", lw=2, ms=4)
    ax[2].set(xlabel="Cable length [km]", ylabel="Minimum compensation [%]",
              title="Compensation needed to hold the voltage band")
    for a in ax: a.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(OUT+"p02_results.png", dpi=160)

## 7. Interpretation and limitations

**What the model captures.** Steady state AC power flow with a distributed shunt
capacitance represented by the standard pi line model, real conductor data, and a
realistic operating voltage band.

**What it does not capture, and why the 160 km figure should not be
quoted as a general number:**

1. Only 110 kV is considered. Real offshore export systems commonly use 220 kV or 275 kV,
   where the same power flows at roughly half the current. Higher voltage extends the AC
   range considerably.
2. The wind farm runs at unity power factor. Modern turbine converters can supply or
   absorb reactive power, which shifts the compensation requirement.
3. Compensation is split evenly between the two ends. Zhu, Dui and Zhang (2015) report
   that offshore side compensation reduces losses more than onshore side compensation,
   so the even split is not optimal.
4. No mid cable compensation platform is considered. Dakic et al. (2021) treat mid cable
   reactors explicitly and that is the natural extension of this study.
5. Cost is not modelled at all. The real AC versus HVDC decision is economic, and the
   technical limit found here is only one input to it.

**How this compares to the literature.** Zhu, Dui and Zhang (2015) state in their
abstract that HVAC is preferred within 40 km of shore. My limit of
160 km is longer, which is consistent with the differences above:
this study loads two circuits to only about 90 percent, so there is
spare conductor capacity to carry charging current that a more heavily loaded design would
not have. The comparison is a reminder that a crossover distance is a property of a
specific design, not a universal constant.

## 8. What I would do next

- Repeat at 220 kV and 275 kV and plot the feasible length against system voltage.
- Add a mid cable reactor and see how much further the AC option reaches.
- Let the wind farm provide reactive support and quantify how much shunt reactor it displaces.
- Attach costs and find the economic crossover rather than the technical one.

## References

1. Thurner, L., Scheidler, A., Schafer, F., Menke, J.-H., Dollichon, J., Meier, F.,
   Meinecke, S., and Braun, M. (2018). "pandapower: An Open-Source Python Tool for
   Convenient Modeling, Analysis, and Optimization of Electric Power Systems."
   *IEEE Transactions on Power Systems*, 33(6), 6510 to 6521.
   DOI: 10.1109/TPWRS.2018.2829021
2. Heuck, K., Dettmann, K.-D., and Schulz, D. (2013). *Elektrische Energieversorgung*,
   9th edition. Springer Vieweg, Wiesbaden. DOI: 10.1007/978-3-8348-2174-4.
   Page 744 is the source pandapower cites for its cable standard types.
3. Zhu, G., Dui, X., and Zhang, C. (2015). "Optimisation of reactive power compensation
   of HVAC cable in off-shore wind power plant." *IET Renewable Power Generation*,
   9(7), 857 to 863. DOI: 10.1049/iet-rpg.2014.0375
4. Dakic, J., Cheah-Mane, M., Gomis-Bellmunt, O., and Prieto-Araujo, E. (2021).
   "HVAC Transmission System for Offshore Wind Power Plants Including Mid-Cable Reactive
   Power Compensation: Optimal Design and Comparison to VSC-HVDC Transmission."
   *IEEE Transactions on Power Delivery*, 36(5), 2814 to 2824.
   DOI: 10.1109/TPWRD.2020.3027356
5. Rahman, S., Khan, I., Alkhammash, H. I., and Nadeem, M. F. (2021). "A Comparison
   Review on Transmission Mode for Onshore Integration of Offshore Wind Farms: HVDC or
   HVAC." *Electronics*, 10(12), 1489. DOI: 10.3390/electronics10121489
6. CIGRE Working Group B1.40 (2015). *Offshore generation cable connections.*
   CIGRE Technical Brochure 610.
